In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PhishGuard AI — RQ2: Preprocessing Documentation & Imbalance Handling
# Input : Stage 0 CSVs (train / val / test) — load only, never re-clean
# Output: rq2_preprocessing/results/
#         • preprocessing_decisions.csv
#         • class_weight_config.csv
#         • experiment_config.csv
# ═══════════════════════════════════════════════════════════════════════════════

# ═══ CELL 1 — Reproducibility Block ═══
import os, random, numpy as np, torch

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

print("✓ Reproducibility block applied (SEED=42)")


# ═══ CELL 2 — Drive mount + folder tree ═══
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR    = "/content/drive/MyDrive/NLP FINAL"
DATASET_DIR = os.path.join(BASE_DIR, "dataset")
STAGE_DIR   = os.path.join(BASE_DIR, "rq2_preprocessing")

for sub in ["models", "logs", "results", "diagrams"]:
    os.makedirs(os.path.join(STAGE_DIR, sub), exist_ok=True)

RESULTS_DIR = os.path.join(STAGE_DIR, "results")

TRAIN_PATH = os.path.join(DATASET_DIR, "train.csv")
VAL_PATH   = os.path.join(DATASET_DIR, "val.csv")
TEST_PATH  = os.path.join(DATASET_DIR, "test.csv")

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing {p}\nRun Stage 0 first.")

print(f"✓ BASE_DIR  : {BASE_DIR}")
print(f"✓ STAGE_DIR : {STAGE_DIR}")


# ═══ CELL 3 — Imports ═══
import pandas as pd
from datetime import datetime
import platform
from sklearn.utils.class_weight import compute_class_weight


# ═══ CELL 4 — Load splits (read-only — never re-clean / re-split) ═══
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert "label" in df.columns, f"{name}.csv missing 'label' column"
    assert set(df["label"].unique()).issubset({0, 1}), f"{name}.csv has invalid labels"

print(f"Loaded → train: {len(train_df):,} | val: {len(val_df):,} | test: {len(test_df):,}")


# ═══ CELL 5 — Compute imbalance statistics from TRAIN set ═══
LABEL_MAP = {0: "Legitimate (Safe Email)", 1: "Phishing (Phishing Email)"}

n_safe     = int((train_df["label"] == 0).sum())
n_phishing = int((train_df["label"] == 1).sum())
n_total    = len(train_df)

pct_safe     = round(n_safe / n_total * 100, 2)
pct_phishing = round(n_phishing / n_total * 100, 2)
imbalance_ratio = round(n_safe / n_phishing, 4)          # majority : minority
minority_class  = 1                                       # Phishing
majority_class  = 0                                       # Safe

# sklearn 'balanced' weights: n_samples / (n_classes * n_samples_per_class)
sklearn_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"].values,
)
weight_class_0 = round(float(sklearn_weights[0]), 6)
weight_class_1 = round(float(sklearn_weights[1]), 6)

# XGBoost: scale_pos_weight = count(negative) / count(positive)  →  safe / phishing
xgb_scale_pos_weight = round(n_safe / n_phishing, 6)

print("\n── Imbalance Summary (Train Set) ──")
print(f"  Legitimate (0) : {n_safe:,}  ({pct_safe}%)")
print(f"  Phishing   (1) : {n_phishing:,}  ({pct_phishing}%)")
print(f"  Ratio (Safe:Phishing) : {imbalance_ratio}:1")
print(f"  sklearn balanced w0={weight_class_0}, w1={weight_class_1}")
print(f"  XGBoost scale_pos_weight = {xgb_scale_pos_weight}")


# ═══ CELL 6 — preprocessing_decisions.csv ═══
print("\nBuilding preprocessing_decisions.csv ...")

preprocessing_decisions = pd.DataFrame([
    # ── General / Stage 0 ──
    {
        "pipeline": "general",
        "step_order": 1,
        "step_name": "raw_data_loading",
        "applied_to": "Phishing_Email.csv",
        "column_produced": "—",
        "description": "Load raw Kaggle CSV from Drive (raw dataset/Phishing_Email.csv).",
        "rationale": "Single authoritative raw source; all downstream stages load processed CSVs only.",
        "stage_implemented": "Stage 0",
    },
    {
        "pipeline": "general",
        "step_order": 2,
        "step_name": "drop_index_column",
        "applied_to": "Unnamed: 0",
        "column_produced": "—",
        "description": "Remove auto-generated Kaggle row index column.",
        "rationale": "Index column carries no predictive signal and would leak row identity if kept.",
        "stage_implemented": "Stage 0",
    },
    {
        "pipeline": "general",
        "step_order": 3,
        "step_name": "remove_null_empty_text",
        "applied_to": "Email Text",
        "column_produced": "—",
        "description": "Drop rows where Email Text is null, NaN, or whitespace-only.",
        "rationale": "Empty emails cannot be classified; reduces dataset from 18,650 → 18,634 rows.",
        "stage_implemented": "Stage 0",
    },
    {
        "pipeline": "general",
        "step_order": 4,
        "step_name": "label_encoding",
        "applied_to": "Email Type",
        "column_produced": "label",
        "description": "Map 'Safe Email' → 0 (Legitimate), 'Phishing Email' → 1 (Phishing).",
        "rationale": "Binary integer labels required by sklearn, XGBoost, and PyTorch loss functions.",
        "stage_implemented": "Stage 0",
    },
    {
        "pipeline": "general",
        "step_order": 5,
        "step_name": "extreme_length_truncation",
        "applied_to": "Email Text",
        "column_produced": "—",
        "description": "Truncate emails exceeding 100,000 characters before cleaning.",
        "rationale": "Removes broken outlier rows (~17M chars) that would exhaust memory in DL/BERT stages.",
        "stage_implemented": "Stage 0",
    },
    {
        "pipeline": "general",
        "step_order": 6,
        "step_name": "stratified_train_val_test_split",
        "applied_to": "full cleaned dataset",
        "column_produced": "train.csv / val.csv / test.csv",
        "description": "70% train / 15% val / 15% test, stratified on label, random_state=42. Done ONCE.",
        "rationale": "Preserves ~60/40 class ratio in every split; split never repeated in later stages.",
        "stage_implemented": "Stage 0",
    },

    # ── Classical pipeline → text_cleaned_classical (RQ4) ──
    {
        "pipeline": "classical",
        "step_order": 1,
        "step_name": "strip_html",
        "applied_to": "Email Text",
        "column_produced": "text_cleaned_classical",
        "description": "Parse and remove HTML tags using BeautifulSoup (lxml parser).",
        "rationale": "HTML markup adds noise tokens ('div', 'br', etc.) that inflate TF-IDF vocabulary without semantic value.",
        "stage_implemented": "Stage 0",
    },
    {
        "pipeline": "classical",
        "step_order": 2,
        "step_name": "url_tokenization",
        "applied_to": "Email Text",
        "column_produced": "text_cleaned_classical",
        "description": "Replace http/https/www URLs with the literal token <URL>.",
        "rationale": "URLs are high-variance features (unique per email); collapsing to one token lets TF-IDF capture 'presence of link' signal.",
        "stage_implemented": "Stage 0",
    },
    {
        "pipeline": "classical",
        "step_order": 3,
        "step_name": "lowercasing",
        "applied_to": "Email Text",
        "column_produced": "text_cleaned_classical",
        "description": "Convert all characters to lowercase.",
        "rationale": "TF-IDF treats 'Account' and 'account' as different tokens; lowercasing reduces vocabulary size and improves term frequency estimates.",
        "stage_implemented": "Stage 0",
    },
    {
        "pipeline": "classical",
        "step_order": 4,
        "step_name": "punctuation_removal",
        "applied_to": "Email Text",
        "column_produced": "text_cleaned_classical",
        "description": "Remove all non-alphabetic characters; keep only a-z and whitespace.",
        "rationale": "Punctuation tokens ('!!!', '$', etc.) are sparse and rarely discriminative for bag-of-words models.",
        "stage_implemented": "Stage 0",
    },
    {
        "pipeline": "classical",
        "step_order": 5,
        "step_name": "stopword_removal",
        "applied_to": "Email Text",
        "column_produced": "text_cleaned_classical",
        "description": "Remove NLTK English stopwords (the, is, at, etc.).",
        "rationale": "Stopwords appear in both classes with similar frequency; removing them highlights content-bearing phishing cues ('click', 'verify', 'password').",
        "stage_implemented": "Stage 0",
    },
    {
        "pipeline": "classical",
        "step_order": 6,
        "step_name": "lemmatization",
        "applied_to": "Email Text",
        "column_produced": "text_cleaned_classical",
        "description": "Reduce words to base form using NLTK WordNetLemmatizer (e.g., 'running' → 'run').",
        "rationale": "Consolidates inflected forms into one token, improving TF-IDF term counts for classical models.",
        "stage_implemented": "Stage 0",
    },
    {
        "pipeline": "classical",
        "step_order": 7,
        "step_name": "whitespace_normalization",
        "applied_to": "Email Text",
        "column_produced": "text_cleaned_classical",
        "description": "Collapse multiple spaces/newlines into a single space; strip leading/trailing whitespace.",
        "rationale": "Clean token boundaries for TF-IDF vectorizer (fit in RQ3).",
        "stage_implemented": "Stage 0",
    },

    # ── Transformer / DL pipeline → text_cleaned_transformer (RQ5, RQ6) ──
    {
        "pipeline": "transformer",
        "step_order": 1,
        "step_name": "strip_html",
        "applied_to": "Email Text",
        "column_produced": "text_cleaned_transformer",
        "description": "Parse and remove HTML tags using BeautifulSoup (lxml parser).",
        "rationale": "Same HTML noise issue as classical pipeline; structure otherwise preserved.",
        "stage_implemented": "Stage 0",
    },
    {
        "pipeline": "transformer",
        "step_order": 2,
        "step_name": "url_tokenization",
        "applied_to": "Email Text",
        "column_produced": "text_cleaned_transformer",
        "description": "Replace http/https/www URLs with the literal token <URL>.",
        "rationale": "Same URL rationale as classical; token is compatible with WordPiece/BPE tokenizers.",
        "stage_implemented": "Stage 0",
    },
    {
        "pipeline": "transformer",
        "step_order": 3,
        "step_name": "preserve_case_punctuation_structure",
        "applied_to": "Email Text",
        "column_produced": "text_cleaned_transformer",
        "description": "NO lowercasing, NO stopword removal, NO lemmatization, NO punctuation removal.",
        "rationale": "DistilBERT WordPiece tokenizer and pretrained weights depend on natural casing ('US' vs 'us'), punctuation cues ('!!!', '$500'), and sentence structure. Heavy cleaning silently degrades transformer performance.",
        "stage_implemented": "Stage 0",
    },
    {
        "pipeline": "transformer",
        "step_order": 4,
        "step_name": "whitespace_normalization",
        "applied_to": "Email Text",
        "column_produced": "text_cleaned_transformer",
        "description": "Collapse excessive whitespace/newlines; strip leading/trailing whitespace.",
        "rationale": "Minor normalization only; sentence structure and word boundaries remain intact for BiLSTM/TextCNN/DistilBERT.",
        "stage_implemented": "Stage 0",
    },

    # ── Dual-pipeline design note ──
    {
        "pipeline": "design_decision",
        "step_order": 1,
        "step_name": "dual_preprocessing_pipelines",
        "applied_to": "all models",
        "column_produced": "text_cleaned_classical + text_cleaned_transformer",
        "description": "Two parallel text columns saved in every split CSV; each model family routes to the correct column.",
        "rationale": "Applying one heavy-cleaning pipeline to all models is the most common student-project mistake. Classical TF-IDF models benefit from aggressive normalization; transformers are harmed by it. This design prevents silent performance loss on DistilBERT/BiLSTM/TextCNN.",
        "stage_implemented": "Stage 0",
    },

    # ── Imbalance handling (formalized in RQ2, applied in RQ4) ──
    {
        "pipeline": "imbalance_handling",
        "step_order": 1,
        "step_name": "class_weight_balanced_sklearn",
        "applied_to": "Logistic Regression, Linear SVM, Random Forest",
        "column_produced": "—",
        "description": f"Instantiate with class_weight='balanced'. Computed weights: class_0={weight_class_0}, class_1={weight_class_1}.",
        "rationale": f"Mild ~{pct_safe}/{pct_phishing} imbalance (Safe:Phishing = {imbalance_ratio}:1). Balanced weights upweight minority Phishing class so models do not bias toward predicting Safe.",
        "stage_implemented": "RQ4 (configured here in RQ2)",
    },
    {
        "pipeline": "imbalance_handling",
        "step_order": 2,
        "step_name": "scale_pos_weight_xgboost",
        "applied_to": "XGBoost",
        "column_produced": "—",
        "description": f"Set scale_pos_weight={xgb_scale_pos_weight} (count_safe / count_phishing = {n_safe}/{n_phishing}).",
        "rationale": "XGBoost has no class_weight param; scale_pos_weight is the equivalent mechanism to penalize missed Phishing (positive class) errors more heavily.",
        "stage_implemented": "RQ4 (configured here in RQ2)",
    },
    {
        "pipeline": "imbalance_handling",
        "step_order": 3,
        "step_name": "naive_bayes_class_prior",
        "applied_to": "Multinomial Naive Bayes",
        "column_produced": "—",
        "description": "MultinomialNB has no class_weight param. Use class_prior=[0.5, 0.5] to treat both classes equally regardless of frequency.",
        "rationale": "Per project ground rules: NB has no class_weight — skip it there. Equal priors prevent the model from inheriting the ~60/40 frequency bias from training data.",
        "stage_implemented": "RQ4 (configured here in RQ2)",
    },
    {
        "pipeline": "imbalance_handling",
        "step_order": 4,
        "step_name": "evaluation_metrics_for_imbalance",
        "applied_to": "all models (RQ8 master table)",
        "column_produced": "—",
        "description": "Report Macro-F1, PR-AUC, and Phishing-class Recall alongside Accuracy and ROC-AUC.",
        "rationale": "Accuracy alone is misleading on imbalanced data. Phishing Recall is operationally critical — a missed phishing email is costlier than a false alarm on legitimate mail.",
        "stage_implemented": "RQ8",
    },
])

decisions_path = os.path.join(RESULTS_DIR, "preprocessing_decisions.csv")
preprocessing_decisions.to_csv(decisions_path, index=False)
print(f"  ✓ Saved: {decisions_path}  ({len(preprocessing_decisions)} rows)")


# ═══ CELL 7 — class_weight_config.csv (RQ4 will load / follow this) ═══
print("\nBuilding class_weight_config.csv ...")

class_weight_config = pd.DataFrame([
    {
        "model_name": "Multinomial Naive Bayes",
        "model_key": "naive_bayes",
        "model_family": "Probabilistic",
        "text_column": "text_cleaned_classical",
        "imbalance_strategy": "equal_class_prior",
        "parameter_name": "class_prior",
        "parameter_value": "[0.5, 0.5]",
        "computed_numeric_value": "0.5, 0.5",
        "formula": "Manual equal priors (no class_weight in MultinomialNB)",
        "sklearn_instantiation_note": "MultinomialNB(class_prior=[0.5, 0.5], fit_prior=False)",
        "used_in_stage": "RQ4",
    },
    {
        "model_name": "Logistic Regression",
        "model_key": "logistic_regression",
        "model_family": "Linear",
        "text_column": "text_cleaned_classical",
        "imbalance_strategy": "sklearn_balanced",
        "parameter_name": "class_weight",
        "parameter_value": "balanced",
        "computed_numeric_value": f"class_0={weight_class_0}, class_1={weight_class_1}",
        "formula": "n_samples / (n_classes * np.bincount(y))",
        "sklearn_instantiation_note": "LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)",
        "used_in_stage": "RQ4",
    },
    {
        "model_name": "Linear SVM",
        "model_key": "linear_svm",
        "model_family": "Margin-based",
        "text_column": "text_cleaned_classical",
        "imbalance_strategy": "sklearn_balanced",
        "parameter_name": "class_weight",
        "parameter_value": "balanced",
        "computed_numeric_value": f"class_0={weight_class_0}, class_1={weight_class_1}",
        "formula": "n_samples / (n_classes * np.bincount(y))",
        "sklearn_instantiation_note": "LinearSVC(class_weight='balanced', random_state=42, max_iter=3000)",
        "used_in_stage": "RQ4",
    },
    {
        "model_name": "Random Forest",
        "model_key": "random_forest",
        "model_family": "Tree ensemble (bagging)",
        "text_column": "text_cleaned_classical",
        "imbalance_strategy": "sklearn_balanced",
        "parameter_name": "class_weight",
        "parameter_value": "balanced",
        "computed_numeric_value": f"class_0={weight_class_0}, class_1={weight_class_1}",
        "formula": "n_samples / (n_classes * np.bincount(y))",
        "sklearn_instantiation_note": "RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=200)",
        "used_in_stage": "RQ4",
    },
    {
        "model_name": "XGBoost",
        "model_key": "xgboost",
        "model_family": "Tree ensemble (boosting)",
        "text_column": "text_cleaned_classical",
        "imbalance_strategy": "scale_pos_weight",
        "parameter_name": "scale_pos_weight",
        "parameter_value": str(xgb_scale_pos_weight),
        "computed_numeric_value": str(xgb_scale_pos_weight),
        "formula": f"count(label=0) / count(label=1) = {n_safe}/{n_phishing}",
        "sklearn_instantiation_note": f"XGBClassifier(scale_pos_weight={xgb_scale_pos_weight}, random_state=42, eval_metric='logloss', verbosity=0)",
        "used_in_stage": "RQ4",
    },
])

# Append dataset-level reference rows at the bottom
class_weight_config = pd.concat([
    class_weight_config,
    pd.DataFrame([{
        "model_name": "── TRAIN SET REFERENCE ──",
        "model_key": "—",
        "model_family": "—",
        "text_column": "—",
        "imbalance_strategy": "—",
        "parameter_name": "train_set_statistics",
        "parameter_value": f"total={n_total}, safe={n_safe}, phishing={n_phishing}",
        "computed_numeric_value": f"safe_pct={pct_safe}%, phishing_pct={pct_phishing}%",
        "formula": f"imbalance_ratio (safe:phishing) = {imbalance_ratio}:1",
        "sklearn_instantiation_note": "Computed from train.csv only (Stage 0 split)",
        "used_in_stage": "RQ2",
    }]),
], ignore_index=True)

config_cw_path = os.path.join(RESULTS_DIR, "class_weight_config.csv")
class_weight_config.to_csv(config_cw_path, index=False)
print(f"  ✓ Saved: {config_cw_path}  ({len(class_weight_config)} rows)")


# ═══ CELL 8 — Split-level class distribution (bonus reference CSV) ═══
print("\nBuilding split_class_distribution.csv (reference for report) ...")

split_rows = []
for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    s = int((df["label"] == 0).sum())
    p = int((df["label"] == 1).sum())
    t = len(df)
    split_rows.append({
        "split": split_name,
        "total": t,
        "legitimate_count": s,
        "phishing_count": p,
        "legitimate_pct": round(s / t * 100, 2),
        "phishing_pct": round(p / t * 100, 2),
        "ratio_safe_to_phishing": round(s / p, 4),
    })

split_dist = pd.DataFrame(split_rows)
split_dist_path = os.path.join(RESULTS_DIR, "split_class_distribution.csv")
split_dist.to_csv(split_dist_path, index=False)
print(f"  ✓ Saved: {split_dist_path}")


# ═══ CELL 9 — experiment_config.csv ═══
config = {
    "project":                  "PhishGuard AI",
    "stage":                    "RQ2 — Preprocessing Documentation & Imbalance Handling",
    "timestamp_utc":            datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
    "seed":                     SEED,
    "cleaning_stage":           "Stage 0 (NOT re-run here)",
    "train_samples":            n_total,
    "legitimate_count":         n_safe,
    "phishing_count":           n_phishing,
    "legitimate_pct":           pct_safe,
    "phishing_pct":             pct_phishing,
    "imbalance_ratio_safe_phishing": imbalance_ratio,
    "sklearn_weight_class_0":   weight_class_0,
    "sklearn_weight_class_1":   weight_class_1,
    "xgboost_scale_pos_weight": xgb_scale_pos_weight,
    "outputs":                  "preprocessing_decisions.csv, class_weight_config.csv, split_class_distribution.csv",
    "python_version":           platform.python_version(),
    "pandas_version":           pd.__version__,
    "numpy_version":            np.__version__,
    "sklearn_version":          __import__("sklearn").__version__,
}
config_df = pd.DataFrame(list(config.items()), columns=["parameter", "value"])
config_path = os.path.join(RESULTS_DIR, "experiment_config.csv")
config_df.to_csv(config_path, index=False)
print(f"  ✓ Saved: {config_path}")


# ═══ CELL 10 — Final summary ═══
print("\n" + "=" * 70)
print("RQ2 — PREPROCESSING DOCUMENTATION & IMBALANCE HANDLING COMPLETE ✓")
print("=" * 70)
print(f"\nImbalance (train): {pct_safe}% Safe / {pct_phishing}% Phishing  ({imbalance_ratio}:1)")
print(f"\nRQ4 will use:")
print(f"  LogReg, SVM, RF  → class_weight='balanced'")
print(f"  XGBoost          → scale_pos_weight={xgb_scale_pos_weight}")
print(f"  Naive Bayes      → class_prior=[0.5, 0.5], fit_prior=False")
print(f"\nResults saved → {RESULTS_DIR}/")
print("  preprocessing_decisions.csv   ← paste into Report Section 4")
print("  class_weight_config.csv         ← paste into Report Section 4 + used by RQ4")
print("  split_class_distribution.csv    ← bonus reference for report")
print("  experiment_config.csv")
print("\nNext step → RQ3 (Feature Engineering: TF-IDF + DL tokenizer)")
print("=" * 70)

display(preprocessing_decisions[["pipeline", "step_name", "rationale"]].head(10))
print("  ...")
display(class_weight_config[["model_name", "parameter_name", "parameter_value", "sklearn_instantiation_note"]])
display(split_dist)

✓ Reproducibility block applied (SEED=42)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ BASE_DIR  : /content/drive/MyDrive/NLP FINAL
✓ STAGE_DIR : /content/drive/MyDrive/NLP FINAL/rq2_preprocessing
Loaded → train: 13,041 | val: 2,795 | test: 2,795

── Imbalance Summary (Train Set) ──
  Legitimate (0) : 7,925  (60.77%)
  Phishing   (1) : 5,116  (39.23%)
  Ratio (Safe:Phishing) : 1.5491:1
  sklearn balanced w0=0.822776, w1=1.274531
  XGBoost scale_pos_weight = 1.549062

Building preprocessing_decisions.csv ...
  ✓ Saved: /content/drive/MyDrive/NLP FINAL/rq2_preprocessing/results/preprocessing_decisions.csv  (22 rows)

Building class_weight_config.csv ...
  ✓ Saved: /content/drive/MyDrive/NLP FINAL/rq2_preprocessing/results/class_weight_config.csv  (6 rows)

Building split_class_distribution.csv (reference for report) ...
  ✓ Saved: /content/drive/MyDrive/NLP FINAL/rq2_preprocessing/results/split_class_di

/tmp/ipykernel_1069/2811070145.py:468: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc":            datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),


,pipeline,step_name,rationale
0,general,raw_data_loading,Single authoritative raw source; all downstrea...
1,general,drop_index_column,Index column carries no predictive signal and ...
2,general,remove_null_empty_text,Empty emails cannot be classified; reduces dat...
3,general,label_encoding,"Binary integer labels required by sklearn, XGB..."
4,general,extreme_length_truncation,Removes broken outlier rows (~17M chars) that ...
5,general,stratified_train_val_test_split,Preserves ~60/40 class ratio in every split; s...
6,classical,strip_html,"HTML markup adds noise tokens ('div', 'br', et..."
7,classical,url_tokenization,URLs are high-variance features (unique per em...
8,classical,lowercasing,TF-IDF treats 'Account' and 'account' as diffe...
9,classical,punctuation_removal,"Punctuation tokens ('!!!', '$', etc.) are spar..."


  ...


,model_name,parameter_name,parameter_value,sklearn_instantiation_note
0,Multinomial Naive Bayes,class_prior,"[0.5, 0.5]","MultinomialNB(class_prior=[0.5, 0.5], fit_prio..."
1,Logistic Regression,class_weight,balanced,"LogisticRegression(class_weight='balanced', ra..."
2,Linear SVM,class_weight,balanced,"LinearSVC(class_weight='balanced', random_stat..."
3,Random Forest,class_weight,balanced,RandomForestClassifier(class_weight='balanced'...
4,XGBoost,scale_pos_weight,1.549062,"XGBClassifier(scale_pos_weight=1.549062, rando..."
5,── TRAIN SET REFERENCE ──,train_set_statistics,"total=13041, safe=7925, phishing=5116",Computed from train.csv only (Stage 0 split)


,split,total,legitimate_count,phishing_count,legitimate_pct,phishing_pct,ratio_safe_to_phishing
0,train,13041,7925,5116,60.77,39.23,1.5491
1,val,2795,1698,1097,60.75,39.25,1.5479
2,test,2795,1699,1096,60.79,39.21,1.5502
